# Project Summary: Masar AI (مسار) — Autonomous Career Guidance Platform

## Project Abstract

Masar AI is an end-to-end multi-agent software platform designed to deliver individualized career counseling, skill gap evaluations, and interactive mentorship for students and early-career professionals. By processing student background attributes alongside uploaded resume documents, the system synthesizes unstructured user input into structured career vectors, actionable skill development roadmaps, and dynamic mentorship interactions.

---

## System Architecture & Technical Specifications

```
  [ Student Inputs ] ---> [ PyPDF2 Engine ]
  (Major, Interests,       (CV / Resume Text
     Experience)              Extraction)
          |                        |
          +-----------+------------+
                      |
                      v
          [ Unified LangChain Pipeline ]
                      |
                      v
             [ Quantized Model ]
           (Qwen2.5-7B Engine)
                      |
          +-----------+-----------+
          |                       |
          v                       v
[ Agent 1 & 2: Assessment ]   [ Agent 3: Mentorship Chat ]
(Career Matches & Roadmaps)   (Context-Aware QA Engine)
          |                       |
          +-----------+-----------+
                      |
                      v
        [ Agent 4: ReportLab Engine ]
        (PDF Compilation & Export)
                      |
                      v
       [ Gradio Microservice Interface ]

```

The application is structured around a modular pipeline divided across eight distinct operational components:

* **Cell 1: System Dependencies & Environment Setup** — Configures core libraries, CUDA acceleration parameters, and environment dependencies.
* **Cell 2: LLM Inference Engine** — Initializes the foundational text-generation pipeline leveraging quantized local execution (`Qwen2.5-7B`).
* **Cell 3: Document Ingestion Engine** — Microservice responsible for extracting, sanitizing, and normalizing text from user-uploaded PDF resumes (`PyPDF2`).
* **Cell 4: Assessment & Skill Gap Engine (Agents 1 & 2)** — Constructs structured LangChain prompt primitives designed to execute single-pass evaluations of candidate skills against current market profiles.
* **Cell 5: Unified Pipeline Orchestration** — Integrates user profile metadata, parsed resume contents, and model parameters into a cohesive execution chain.
* **Cell 6: Contextual Mentorship Engine (Agent 3)** — Powers a stateful conversational agent capable of handling follow-up queries while maintaining reference to the user's primary career evaluation.
* **Cell 7: Document Compilation Microservice (Agent 4)** — Converts raw output logs and conversational state into structured PDF documents utilizing a PLATYPUS page-budget design (`ReportLab`).
* **Cell 8: Gateway & User Interface** — Deploys a two-tier Gradio web interface handling event queues, token-level streaming, and session state persistence.

---

## Core Technical Features

| Feature | Implementation Mechanism | Technical Objective |
| --- | --- | --- |
| **Document Processing** | PyPDF2 Text Extraction | Normalizes unstructured PDF input for prompt injection. |
| **Real-Time Streaming** | LangChain `stream` Interface | Minimizes user latency by displaying token outputs live. |
| **Context Retention** | Session State Orchestration | Maintains evaluation history across the chat interface. |
| **Report Generation** | ReportLab PLATYPUS Layout | Compiles styled PDF reports containing assessment logs. |

In [ ]:
# Cell 1: Environment Setup & Service Dependencies
!pip install -q -U transformers accelerate bitsandbytes langchain langchain-huggingface pypdf reportlab gradio

print("✅ All dependencies successfully installed and verified!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.0 MB/s eta 0:00:00
✅ All dependencies successfully installed and verified!


In [ ]:
# Cell 2: Optimized Model Service Initialization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. Verify CUDA / GPU Availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Compute Hardware: {device.upper()}")

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU Device: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ WARNING: No GPU detected! Please change notebook runtime to GPU (T4).")

# 2. Configure 4-bit NF4 Quantization Settings
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Model Repository Identifier
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"🔄 Loading model '{MODEL_ID}' with 4-bit quantization and SDPA acceleration...")

# Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"  # Fast PyTorch Scaled Dot-Product Attention
)

# 4. Create HuggingFace Text Generation Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.4,
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

# Wrap in LangChain Pipeline
llm = HuggingFacePipeline(pipeline=pipe)

print("✅ Model Service successfully initialized and optimized!")

🖥️ Compute Hardware: CUDA
🎮 GPU Device: Tesla T4 (15.64 GB VRAM)
🔄 Loading model 'Qwen/Qwen2.5-7B-Instruct' with 4-bit quantization and SDPA acceleration...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'repetition_penalty', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model Service successfully initialized and optimized!


In [ ]:
# Cell 3: Ingestion Microservice (PDF Resume Extractor)
import pypdf

def extract_text_from_pdf(pdf_file) -> str:
    """
    Ingestion Helper: Extracts raw text content from an uploaded PDF file.
    Truncates large text to prevent prompt overflow.
    """
    if pdf_file is None:
        return ""

    try:
        file_path = pdf_file.name if hasattr(pdf_file, "name") else pdf_file
        reader = pypdf.PdfReader(file_path)
        extracted_text = ""

        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                extracted_text += f"\n--- Page {page_num + 1} ---\n" + page_text

        cleaned_text = extracted_text.strip()

        # Keep within reasonable token limit for performance
        if len(cleaned_text) > 3000:
            cleaned_text = cleaned_text[:3000] + "\n...[Truncated for brevity]"

        print(f"📄 Resume Ingestion Successful: Extracted {len(cleaned_text)} characters.")
        return cleaned_text

    except Exception as e:
        print(f"⚠️ Error reading PDF file: {str(e)}")
        return ""

print("✅ Ingestion Microservice (PDF Extractor) initialized successfully!")

✅ Ingestion Microservice (PDF Extractor) initialized successfully!


In [ ]:
# Cell 4 & 5: Combined Assessment Microservice (Unified Fast Chain)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Combined prompt: Generates Profile Summary AND Industry Gap Analysis in 1 single pass
unified_assessment_template = """You are Masar AI (مسار), an expert career guidance system.
Based on the student's inputs, create a structured evaluation containing two clear sections: Student Profile Summary and Industry Alignment & Gap Analysis.

### Student Inputs:
- Academic Major / Background: {major}
- What they enjoy or are curious about: {interests}
- General Skill Level: {skill_level}
- Extracted Resume / CV Content: {cv_text}

---
### Output Format Requirements:

# 👤 PART 1: STUDENT PROFILE SUMMARY
1. **Current Background**: Summarize their education or current stage.
2. **Exploratory Interests**: Highlight what excites them or what they want to learn.
3. **Core Strengths & Potential**: Identify potential strengths based on their inputs.

---

# 🎯 PART 2: CAREER DIRECTIONS & GAP ANALYSIS
1. **Top 3 Recommended Career Directions**:
   - Provide 3 distinct, highly tailored career paths. Explain *why* each fits them.
2. **Skill Gap Analysis**:
   - Identify specific technical skills, modern tools, or industry frameworks they are missing.
   - Highlight any soft skill or domain knowledge gaps.
3. **High-Impact Development Priorities**:
   - List 3 to 4 immediate, actionable priorities to bridge these skill gaps.
"""

unified_prompt = PromptTemplate(
    template=unified_assessment_template,
    input_variables=["major", "interests", "skill_level", "cv_text"]
)

unified_chain = unified_prompt | llm | StrOutputParser()

print("✅ Combined Unified Assessment Microservice initialized successfully!")

✅ Combined Unified Assessment Microservice initialized successfully!


In [ ]:
# Cell 6: Mentorship Chat Microservice (Final Anti-Simulation Polish)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

agent3_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are Masar AI (مسار), an expert and concise career mentor.

### Student Background Context:
{assessment_context}

---
### STRICT BEHAVIORAL RULES:
1. Address the student directly. Never use placeholders like [Student Name].
2. **CRITICAL:** Output ONLY your direct answer to the user's message.
3. **NEVER** simulate the user, write "Hello, I am...", or invent a fake follow-up question for the user.
4. Keep responses structured, professional, and under 250 words. Stop immediately after your closing sentence.
"""),
    ("human", "{input}")
])

# Expanded stop tokens to catch any attempt at roleplaying both sides
safe_llm = llm.bind(stop=["Human:", "User:", "---", "assistant:", "Hello, I am"])

# Build the chain
agent3_chain = agent3_prompt | safe_llm | StrOutputParser()

def run_agent3_chat_stream(user_message: str, assessment_context: str):
    """Streams chat responses cleanly with aggressive anti-simulation filtering."""
    safe_context = assessment_context if assessment_context and assessment_context.strip() else "Student Profile: Name unknown."

    try:
        for chunk in agent3_chain.stream({
            "input": user_message,
            "assessment_context": safe_context
        }):
            # Live filter to instantly drop any leaked simulation text
            if any(marker in chunk for marker in ["Human:", "User:", "Hello, I am"]):
                break

            yield chunk

    except Exception as e:
        yield f"⚠️ Mentorship Chat Error: {str(e)}"

print("✅ Cell 6 updated with final anti-simulation guards!")

✅ Cell 6 updated with final anti-simulation guards!


In [ ]:
import os
import re
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

def sanitize_text_for_reportlab(text):
    """Clean markdown and HTML to avoid ReportLab platypus parsing errors."""
    if not text:
        return ""
    # Convert newlines to HTML break tags for ReportLab
    text = str(text).replace("\n", "<br/>")

    # Clean up standard Markdown bold syntax (**text** -> <b>text</b>)
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)

    # Remove any unclosed single angle brackets that cause XML syntax errors
    text = re.sub(r'<(?!/?(b|i|u|br/|font)[ >])', '&lt;', text)

    return text

def generate_pdf_report(student_name, assessment_text, chat_history):
    pdf_filename = "Masar_AI_Career_Report.pdf"

    # 1. Page Setup
    doc = SimpleDocTemplate(
        pdf_filename,
        pagesize=letter,
        rightMargin=40, leftMargin=40,
        topMargin=40, bottomMargin=40
    )

    styles = getSampleStyleSheet()

    # 2. Styles
    title_style = ParagraphStyle(
        'DocTitle', parent=styles['Heading1'],
        fontName='Helvetica-Bold', fontSize=22, leading=26,
        textColor=colors.HexColor('#1E3A8A'), spaceAfter=6
    )
    h1_style = ParagraphStyle(
        'Heading1Custom', parent=styles['Heading2'],
        fontName='Helvetica-Bold', fontSize=14, leading=18,
        textColor=colors.HexColor('#1E40AF'), spaceBefore=12, spaceAfter=6
    )
    body_style = ParagraphStyle(
        'BodyCustom', parent=styles['Normal'],
        fontName='Helvetica', fontSize=9.5, leading=14,
        textColor=colors.HexColor('#1F2937'), spaceAfter=6
    )

    story = []

    # --- Header ---
    safe_name = sanitize_text_for_reportlab(student_name if student_name else "Student")
    story.append(Paragraph(f"Masar AI — Career & Mentorship Report", title_style))
    story.append(Paragraph(f"<b>Candidate:</b> {safe_name}", body_style))
    story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#1E3A8A'), spaceAfter=12))

    # --- Section 1: Assessment Report ---
    story.append(Paragraph("1. Career Assessment & Skill Gap Roadmap", h1_style))
    if assessment_text:
        clean_assessment = sanitize_text_for_reportlab(assessment_text)
        story.append(Paragraph(clean_assessment, body_style))
    else:
        story.append(Paragraph("<i>No assessment data recorded.</i>", body_style))

    story.append(Spacer(1, 10))

    # --- Section 2: Mentorship Chat Transcript ---
    story.append(Paragraph("2. Interactive Mentorship Q&A Transcript", h1_style))

    # Handle chat history formats cleanly
    if chat_history and len(str(chat_history).strip()) > 0:
        clean_chat = sanitize_text_for_reportlab(chat_history)
        story.append(Paragraph(clean_chat, body_style))
    else:
        story.append(Paragraph("<i>No Q&A mentorship session was recorded for this report.</i>", body_style))

    # Build PDF safely
    try:
        doc.build(story)
        return pdf_filename
    except Exception as e:
        print(f"Error generating PDF: {e}")
        # Fallback build with plain text if styling fails
        return build_fallback_pdf(pdf_filename, safe_name, assessment_text, chat_history)

def build_fallback_pdf(filename, name, assessment, chat):
    """Safeguard function to create a basic PDF if styled formatting fails."""
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [
        Paragraph(f"Masar AI Report - {name}", styles['Heading1']),
        Spacer(1, 10),
        Paragraph("Assessment Summary:", styles['Heading2']),
        Paragraph(str(assessment).replace('\n', '<br/>'), styles['Normal']),
        Spacer(1, 10),
        Paragraph("Chat Transcript:", styles['Heading2']),
        Paragraph(str(chat).replace('\n', '<br/>'), styles['Normal'])
    ]
    doc.build(story)
    return filename

In [ ]:
# Cell 8: Gateway & Frontend Microservice (Original Stable Version)
import gradio as gr

def process_assessment_stream(name, major, interests, skill_level, cv_file):
    """Streams single-pass assessment live while locking student identity into state."""
    if not major and not interests:
        yield "⚠️ Please enter at least a major or some interests.", "⚠️ Missing inputs.", ""
        return

    cv_text = extract_text_from_pdf(cv_file) if cv_file else "No CV provided."
    if not cv_text:
        cv_text = "No CV provided."

    user_name = name.strip() if name and name.strip() else "Student"
    major_str = major.strip() if major and major.strip() else "Undecided / General Student"
    interests_str = interests.strip() if interests and interests.strip() else "Exploring open options"
    skill_level_str = skill_level if skill_level else "Beginner / Exploring"

    accumulated_text = ""
    status = "⏳ Generating your career report live..."

    for chunk in unified_chain.stream({
        "major": major_str,
        "interests": interests_str,
        "skill_level": skill_level_str,
        "cv_text": cv_text
    }):
        accumulated_text += chunk
        yield accumulated_text, status, accumulated_text

    final_status = f"✅ Assessment completed for {user_name}! Proceed to Tab 2 to chat with your mentor."
    yield accumulated_text, final_status, accumulated_text


def process_chat_stream(user_msg, chat_history, assessment_state):
    """Streams chat responses smoothly token-by-token using standard Gradio message format."""
    if not user_msg or not user_msg.strip():
        yield "", chat_history
        return

    if chat_history is None:
        chat_history = []

    updated_history = list(chat_history)
    updated_history.append({"role": "user", "content": user_msg})
    updated_history.append({"role": "assistant", "content": ""})
    yield "", updated_history

    bot_response = ""
    for token in run_agent3_chat_stream(user_msg, assessment_state):
        bot_response += token
        updated_history[-1] = {"role": "assistant", "content": bot_response}
        yield "", updated_history


def process_pdf_download(name, assessment_state, chat_history):
    roadmap_text = ""
    if chat_history:
        user_q = ""
        for item in chat_history:
            if isinstance(item, dict):
                role = item.get("role")
                content = item.get("content", "")
                if role == "user":
                    user_q = content
                elif role == "assistant" and user_q:
                    roadmap_text += f"<b>Q: {user_q}</b><br/>{content}<br/><br/>"
                    user_q = ""
            elif isinstance(item, (list, tuple)) and len(item) == 2:
                q, a = item[0], item[1]
                if q and a:
                    roadmap_text += f"<b>Q: {q}</b><br/>{a}<br/><br/>"

    pdf_path = generate_pdf_report(name, assessment_state, chat_history)
    return pdf_path


theme = gr.themes.Soft(primary_hue="blue", secondary_hue="indigo", neutral_hue="slate")

with gr.Blocks(theme=theme, title="Masar AI - Career Guidance Platform") as demo:
    assessment_state = gr.State("")

    gr.Markdown("# 🚀 Masar AI (مسار) — Career Guidance System")

    with gr.Tabs():
        with gr.TabItem("📋 Step 1: Tell Us About Yourself"):
            gr.Markdown("### Simple Background Survey")

            with gr.Row():
                with gr.Column(scale=1):
                    student_name = gr.Textbox(label="Your Name", placeholder="e.g., Ali")
                    major_input = gr.Textbox(
                        label="What are you studying? (or recently graduated from)",
                        placeholder="e.g., Computer Science, Business, Arts, High School"
                    )
                    interests_input = gr.Textbox(
                        label="What subjects, hobbies, or fields excite you?",
                        placeholder="e.g., I love watching movies and gym, music, and data science...",
                        lines=3
                    )
                    skill_level_input = gr.Dropdown(
                        choices=["Complete Beginner", "Intermediate / Learning", "Advanced"],
                        value="Complete Beginner",
                        label="Your Current Experience Level"
                    )
                    cv_upload = gr.File(label="Upload Resume/CV (Optional PDF)", file_types=[".pdf"])

                    submit_btn = gr.Button("🚀 Explore My Career Matches", variant="primary")
                    status_output = gr.Markdown("")

                with gr.Column(scale=1):
                    assessment_output = gr.Markdown("### 🎯 Your Assessment & Career Roadmap\n*Fill the form and click submit to watch your recommendations generate live.*")

            submit_btn.click(
                fn=process_assessment_stream,
                inputs=[student_name, major_input, interests_input, skill_level_input, cv_upload],
                outputs=[assessment_output, status_output, assessment_state]
            )

        with gr.TabItem("💬 Step 2: Mentorship Chat"):
            gr.Markdown("### Chat with Your Career Mentor")

            chatbot = gr.Chatbot(height=400, label="Masar Career Mentor")

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Ask anything! e.g., 'How do I combine music and data science?'...",
                    label="Your Question",
                    scale=4
                )
                send_btn = gr.Button("Send 💬", variant="primary", scale=1)

            with gr.Row():
                clear_chat_btn = gr.Button("🗑️ Clear Chat")
                pdf_btn = gr.Button("📄 Download PDF Career Report", variant="secondary")

            pdf_file_output = gr.File(label="Download PDF Report", interactive=False)

            send_btn.click(
                fn=process_chat_stream,
                inputs=[msg_input, chatbot, assessment_state],
                outputs=[msg_input, chatbot]
            )

            msg_input.submit(
                fn=process_chat_stream,
                inputs=[msg_input, chatbot, assessment_state],
                outputs=[msg_input, chatbot]
            )

            clear_chat_btn.click(lambda: None, None, chatbot, queue=False)

            pdf_btn.click(
                fn=process_pdf_download,
                inputs=[student_name, assessment_state, chatbot],
                outputs=[pdf_file_output]
            )

print("✅ Gateway & Frontend Microservice updated successfully!")
demo.queue().launch(share=True, debug=True)

/tmp/ipykernel_9283/2652242162.py:80: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Masar AI - Career Guidance Platform") as demo:


✅ Gateway & Frontend Microservice updated successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7b8fec53d264d518c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7b8fec53d264d518c9.gradio.live
